<a href="https://colab.research.google.com/github/lizlorenzana931109-art/finpredict-nlp/blob/main/00_dataset_generation_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinPredict NLP
## 00 - Generación y validación del dataset

Este notebook documenta el proceso reproducible de generación y validación
del dataset sintético utilizado en **FinPredict NLP**.

El objetivo del dataset es proporcionar un conjunto controlado de textos
relacionados con la gestión de créditos hipotecarios para evaluar modelos
preentrenados de procesamiento de lenguaje natural en la identificación
temprana de señales de riesgo financiero.

### Flujo del notebook

1. Configuración del entorno.
2. Ejecución del script de generación.
3. Carga del dataset.
4. Validación de dimensiones.
5. Validación de valores nulos.
6. Validación de duplicados.
7. Distribución de clases.
8. Distribución de categorías.
9. Análisis de dificultad lingüística.
10. Validación de lenguaje local.
11. Conclusiones de calidad del dataset.

## 1. Configuración del entorno

El código fuente para la generación del dataset se encuentra versionado
en el repositorio de GitHub del proyecto.

En esta sección se clona el repositorio para garantizar que el notebook
utilice la misma versión del script documentada en el proyecto.

In [1]:
# Clonar el repositorio del proyecto
!git clone https://github.com/lizlorenzana931109-art/finpredict-nlp.git

Cloning into 'finpredict-nlp'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 58 (delta 21), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 66.25 KiB | 2.76 MiB/s, done.
Resolving deltas: 100% (21/21), done.


In [2]:
%cd finpredict-nlp

/content/finpredict-nlp


## 2. Ejecución del script de generación

El dataset sintético se genera mediante el script:

`src/data_generation.py`

Este script contiene las reglas de composición, balance de clases,
variaciones lingüísticas y criterios definidos para construir el dataset
de FinPredict NLP de forma reproducible.

In [3]:
# Ejecutar el script de generación del dataset
!python src/data_generation.py

Dataset generado correctamente.
Total de registros: 2000

Distribución por clase:
clase
SIN_SEÑAL_RIESGO    1000
SEÑAL_RIESGO        1000
Name: count, dtype: int64

Archivo guardado en: data/finpredict_nlp_dataset.csv


## 3. Carga del dataset

Una vez generado el archivo, se carga mediante pandas para realizar
las validaciones de calidad y composición.

In [5]:
import pandas as pd

df = pd.read_csv("data/finpredict_nlp_dataset.csv")

print("Dimensiones del dataset:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())

display(df.head(10))

Dimensiones del dataset: (2000, 7)

Columnas:
['id', 'texto', 'etiqueta', 'clase', 'tipo_senal', 'dificultad', 'expresion_local']


,id,texto,etiqueta,clase,tipo_senal,dificultad,expresion_local
0,FP0001,Quería comentar que hice una transferencia par...,0,SIN_SEÑAL_RIESGO,confirmacion_pago,baja,0
1,FP0002,Necesito informar que quiero saber cuánto sald...,0,SIN_SEÑAL_RIESGO,consulta_informacion,baja,0
2,FP0003,Quería comentar que no cuento con suficiente e...,1,SEÑAL_RIESGO,problemas_liquidez,media,0
3,FP0004,Quería comentar que tuvimos gastos familiares ...,1,SEÑAL_RIESGO,dificultades_familiares,baja,0
4,FP0005,Necesito informar que por ahora no tengo sufic...,1,SEÑAL_RIESGO,problemas_liquidez,alta,0
5,FP0006,Actualmente me está costando cubrir la mensual...,1,SEÑAL_RIESGO,dificultad_pago,media,0
6,FP0007,Necesito informar que necesito revisar el sald...,0,SIN_SEÑAL_RIESGO,estado_cuenta,media,0
7,FP0008,Actualmente necesito posponer temporalmente el...,1,SEÑAL_RIESGO,diferimiento,media,0
8,FP0009,Por el momento quiero consultar el detalle de ...,0,SIN_SEÑAL_RIESGO,estado_cuenta,baja,0
9,FP0010,Quería checar algo: por el momento quisiera ac...,0,SIN_SEÑAL_RIESGO,actualizacion_datos,media,1


## 4. Validación de dimensiones

Se verifica que el dataset generado contenga los 2,000 registros definidos
en el diseño experimental y las siete variables documentadas.

In [6]:
# Validación de dimensiones
filas, columnas = df.shape

print(f"Total de registros: {filas}")
print(f"Total de columnas: {columnas}")

assert filas == 2000, "El dataset no contiene 2,000 registros."
assert columnas == 7, "El número de columnas no coincide con lo esperado."

print("\n✅ Dimensiones correctas.")

Total de registros: 2000
Total de columnas: 7

✅ Dimensiones correctas.


## 5. Validación de valores nulos

Se comprueba que ninguna de las variables utilizadas en el experimento
contenga valores faltantes.

In [7]:
# Validación de valores nulos
nulos = df.isnull().sum()

display(nulos.to_frame(name="valores_nulos"))

assert nulos.sum() == 0, "Se encontraron valores nulos en el dataset."

print("✅ No se encontraron valores nulos.")

,valores_nulos
id,0
texto,0
etiqueta,0
clase,0
tipo_senal,0
dificultad,0
expresion_local,0


✅ No se encontraron valores nulos.


## 6. Validación de textos duplicados

Se verifica que no existan textos idénticos dentro del dataset, con el
objetivo de evitar observaciones repetidas que puedan afectar la evaluación
posterior de los modelos.

In [9]:
# Validación de duplicados
duplicados = df["texto"].duplicated().sum()

print(f"Textos duplicados: {duplicados}")

assert duplicados == 0, "Se encontraron textos duplicados."

print("✅ No se encontraron textos duplicados.")

Textos duplicados: 0
✅ No se encontraron textos duplicados.


## 7. Distribución de clases

El problema se diseñó como una clasificación binaria balanceada.

La distribución esperada es:

- 1,000 registros `SEÑAL_RIESGO`.
- 1,000 registros `SIN_SEÑAL_RIESGO`.

Este balance permite comparar los modelos bajo condiciones controladas y
evita que una clase domine las métricas globales.

In [10]:
# Distribución de clases
distribucion_clases = df["clase"].value_counts()

display(
    distribucion_clases
    .rename_axis("clase")
    .reset_index(name="registros")
)

assert distribucion_clases["SEÑAL_RIESGO"] == 1000
assert distribucion_clases["SIN_SEÑAL_RIESGO"] == 1000

print("✅ Dataset balanceado 50/50.")

,clase,registros
0,SIN_SEÑAL_RIESGO,1000
1,SEÑAL_RIESGO,1000


✅ Dataset balanceado 50/50.


## 8. Distribución por categorías

El dataset contiene distintos tipos de interacción y señales financieras
para evitar que la clasificación dependa de un único patrón de lenguaje.

Se valida que la distribución por categoría coincida con el diseño definido
previamente en `data/README.md`.

In [11]:
# Distribución por tipo de señal
distribucion_categorias = (
    df["tipo_senal"]
    .value_counts()
    .rename_axis("tipo_senal")
    .reset_index(name="registros")
)

display(distribucion_categorias)

,tipo_senal,registros
0,confirmacion_pago,180
1,aclaracion_pago,150
2,consulta_informacion,140
3,estado_cuenta,140
4,problemas_liquidez,140
5,perdida_empleo,140
6,reduccion_ingresos,140
7,reestructura,120
8,dificultad_pago,120
9,situacion_hipotetica,120


## 9. Análisis de dificultad lingüística

La dificultad lingüística no se asigna de forma aleatoria.

Cada texto se clasifica como `baja`, `media` o `alta` a partir de
características observables como:

- negación;
- temporalidad;
- situaciones hipotéticas;
- situaciones resueltas;
- incertidumbre;
- lenguaje indirecto;
- expresiones locales.

Esto permitirá analizar posteriormente si el desempeño de los modelos
disminuye en textos con mayor complejidad contextual.

In [12]:
# Distribución por dificultad
distribucion_dificultad = (
    df["dificultad"]
    .value_counts()
    .rename_axis("dificultad")
    .reset_index(name="registros")
)

distribucion_dificultad["porcentaje"] = (
    distribucion_dificultad["registros"] / len(df) * 100
).round(2)

display(distribucion_dificultad)

,dificultad,registros,porcentaje
0,media,859,42.95
1,alta,653,32.65
2,baja,488,24.40


## 10. Validación de lenguaje local

El dataset incluye expresiones y construcciones contextualizadas al español
utilizado en México.

Para evitar que el uso de lenguaje local quede asociado artificialmente con
una sola clase, se estableció una distribución balanceada:

- 200 registros `SEÑAL_RIESGO`.
- 200 registros `SIN_SEÑAL_RIESGO`.

En total, 400 registros contienen lenguaje local.

In [13]:
# Lenguaje local por clase
tabla_local = pd.crosstab(
    df["clase"],
    df["expresion_local"]
)

display(tabla_local)

assert tabla_local.loc["SEÑAL_RIESGO", 1] == 200
assert tabla_local.loc["SIN_SEÑAL_RIESGO", 1] == 200

print("✅ Lenguaje local balanceado entre clases.")

expresion_local,0,1
clase,,
SEÑAL_RIESGO,800,200
SIN_SEÑAL_RIESGO,800,200


✅ Lenguaje local balanceado entre clases.


### Ejemplos de lenguaje local

Se revisan algunos registros para comprobar que la variable
`expresion_local` corresponde efectivamente con diferencias lingüísticas
observables.

In [14]:
ejemplos_locales = (
    df[df["expresion_local"] == 1]
    [["texto", "clase", "tipo_senal", "dificultad"]]
    .sample(n=10, random_state=42)
)

display(ejemplos_locales)

,texto,clase,tipo_senal,dificultad
1043,Nomás para comentar: por el momento estuve des...,SIN_SEÑAL_RIESGO,situacion_resuelta,alta
1387,Nomás para comentar: actualmente necesito actu...,SIN_SEÑAL_RIESGO,actualizacion_datos,alta
183,Nomás para comentar: necesito informar que qui...,SIN_SEÑAL_RIESGO,situacion_hipotetica,alta
1048,Quería checar algo: en este momento me quedé s...,SEÑAL_RIESGO,perdida_empleo,baja
479,En este momento creo que este mes me voy a atr...,SEÑAL_RIESGO,dificultad_pago,alta
434,Quería comentar que me quedé sin chamba y me p...,SEÑAL_RIESGO,perdida_empleo,alta
1642,Quería checar algo: por el momento necesito co...,SIN_SEÑAL_RIESGO,consulta_preventiva,alta
480,Quería checar algo: actualmente surgió una eme...,SEÑAL_RIESGO,gastos_extraordinarios,baja
1326,Quería checar algo: tuve una reducción tempora...,SIN_SEÑAL_RIESGO,situacion_resuelta,alta
669,En este momento creo que este mes me voy a atr...,SEÑAL_RIESGO,dificultad_pago,alta


## 11. Conclusión de calidad del dataset

El proceso de validación confirma que el dataset utilizado en FinPredict NLP
cumple con las condiciones definidas para la experimentación.

Resultados principales:

- 2,000 registros sintéticos.
- 7 variables documentadas.
- 0 valores nulos.
- 0 textos duplicados.
- 1,000 registros con señal de riesgo.
- 1,000 registros sin señal de riesgo.
- múltiples categorías de interacción.
- dificultad lingüística determinada mediante reglas contextuales.
- 400 registros con lenguaje local.
- representación balanceada del lenguaje local entre ambas clases.

Con base en estas validaciones, el dataset se considera adecuado para iniciar
la evaluación comparativa de modelos preentrenados de Hugging Face.

El archivo generado corresponde a:

`data/finpredict_nlp_dataset.csv`

## 12. Descarga del dataset validado

Una vez completadas las validaciones de calidad, el dataset puede
descargarse en formato CSV para su conservación o revisión externa.

El archivo corresponde a la versión generada y validada durante la
ejecución de este notebook.

In [16]:
from google.colab import files

files.download("data/finpredict_nlp_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>